# Pipeline dịch manhwa — bản gộp toàn bộ (chạy 1 mạch)

Notebook này gộp toàn bộ 6 bước của pipeline dịch manhwa thành **1 file duy nhất**, chạy tuần tự từ trên xuống dưới:

1. Tải ảnh chapter từ MangaDex
2. Phát hiện bong bóng thoại (YOLOv8) + OCR từng bong bóng (PaddleOCR) — chạy offline trong Colab, không cần API key
3. Dịch Anh → Việt (Groq API, LLM `openai/gpt-oss-120b`)
4. Làm sạch vùng chữ (lọc tên chương/logo)
5. Xóa chữ trong ảnh (LaMa inpainting)
6. Giao diện Gradio để bạn sửa bản dịch + chèn chữ Việt vào ảnh

**Cách dùng:** đổi `CHAPTER_ID` ở cell cấu hình chung ngay dưới đây (chỉ cần đổi 1 chỗ duy nhất, không phải sửa nhiều nơi như khi dùng 6 notebook riêng), sau đó **Runtime → Run all** (hoặc chạy từng cell tuần tự). Notebook sẽ tự động chạy hết các bước 1-5, dừng lại ở bước 6 (Gradio) — lúc đó bạn thao tác sửa bản dịch trên giao diện web hiện ra.

**Lưu ý về thời gian chạy:** bước 5 (LaMa) cần tạo venv riêng, mất vài phút để cài đặt mỗi phiên Colab mới. Bước 2 (YOLOv8 + PaddleOCR) cũng cần cài đặt/tải model lần đầu (vài phút, model YOLOv8 chỉ tải 1 lần rồi lưu vào Drive), các lần chạy OCR sau trong cùng phiên sẽ nhanh hơn. Bước 3 (Groq) chỉ là HTTP call nên khá nhanh. Tổng thời gian chạy hết bước 1-5 cho 1 chapter thường khoảng 10-20 phút tùy độ dài chapter.

**Nếu cần chạy lại từ giữa chừng** (VD chapter đã tải + OCR xong, chỉ muốn dịch lại): có thể chạy riêng lẻ từng section bằng cách nhảy tới cell tương ứng — mọi biến cấu hình (`PROJECT_ROOT`, `CHAPTER_ID`, các thư mục) đều được định nghĩa ngay từ đầu nên các section phía sau vẫn chạy độc lập được, miễn là cell cấu hình chung đã chạy trước đó trong cùng phiên Colab.

**Mục lục:**
- BƯỚC 0 — Cấu hình chung (Drive + `CHAPTER_ID`)
- BƯỚC 1 — Tải ảnh chapter từ MangaDex
- BƯỚC 2 — Phát hiện bong bóng (YOLOv8) + OCR từng bong bóng (PaddleOCR)
- BƯỚC 3 — Dịch Anh → Việt (Groq API)
- BƯỚC 4 — Làm sạch vùng chữ (lọc tên chương/logo)
- BƯỚC 5 — Xóa chữ trong ảnh (LaMa inpainting)
- BƯỚC 6 — Giao diện Gradio (sửa bản dịch + chèn chữ Việt)

## 0. Cấu hình chung — chạy DUY NHẤT 1 lần cho toàn bộ pipeline

Kết nối Google Drive, khai báo `CHAPTER_ID` và toàn bộ đường dẫn thư mục con dùng chung cho tất cả các bước bên dưới.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/manga_translation_tool'
os.makedirs(PROJECT_ROOT, exist_ok=True)

# ĐỔI CHAPTER_ID Ở ĐÂY — đây là nơi DUY NHẤT cần sửa khi chuyển sang
# chapter khác, mọi bước phía dưới đều dùng chung biến này.
CHAPTER_ID = "c03cb6f1-00ad-481c-b481-ac2cea2dfb17"

# Đường dẫn các thư mục con, dùng xuyên suốt toàn bộ pipeline
RAW_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'raw')
OCR_OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'ocr_results')
BACKUP_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'ocr_results_backup_before_merge')
INPAINTED_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'inpainted')
MASK_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'masks')
FINAL_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'final')

for d in (RAW_DIR, OCR_OUTPUT_DIR, BACKUP_DIR, INPAINTED_DIR, MASK_DIR, FINAL_DIR):
    os.makedirs(d, exist_ok=True)

print(f'Thư mục dự án: {PROJECT_ROOT}')
print(f'Chapter đang xử lý: {CHAPTER_ID}')

---
# BƯỚC 1: Tải ảnh chapter từ MangaDex

## 1.1 Cài thư viện cần thiết

In [ ]:
!pip install -q requests tqdm

## 1.2 Thiết lập thư mục lưu ảnh

`OUTPUT_DIR` dùng chung với `RAW_DIR` đã khai báo ở mục 0.

In [ ]:
USE_ORIGINAL_QUALITY = True  # True = ảnh chất lượng gốc (khuyên dùng, OCR cần đọc chữ rõ)

OUTPUT_DIR = RAW_DIR
print(f'Ảnh sẽ được lưu tại: {OUTPUT_DIR}')

## 1.3 Lấy thông tin server ảnh

Lưu ý: `baseUrl` trả về chỉ có hiệu lực khoảng 15 phút, nên các bước tải ảnh phía dưới cần chạy ngay sau bước này, không nên để cách quãng lâu.

In [ ]:
import requests

def get_chapter_server_info(chapter_id: str) -> dict:
    """Gọi MangaDex API để lấy base URL + danh sách trang của chapter."""
    url = f"https://api.mangadex.org/at-home/server/{chapter_id}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return resp.json()

server_info = get_chapter_server_info(CHAPTER_ID)

base_url = server_info["baseUrl"]
chapter_hash = server_info["chapter"]["hash"]
pages = server_info["chapter"]["data"] if USE_ORIGINAL_QUALITY else server_info["chapter"]["dataSaver"]
quality_folder = "data" if USE_ORIGINAL_QUALITY else "data-saver"

print(f"Base URL: {base_url}")
print(f"Chapter hash: {chapter_hash}")
print(f"Số trang: {len(pages)}")
print(f"Trang đầu tiên: {pages[0] if pages else 'Không có dữ liệu'}")

## 1.4 Tải toàn bộ ảnh + báo cáo về MangaDex

Mỗi ảnh tải xong sẽ được báo cáo (report) về `api.mangadex.network/report` — đây là bước bắt buộc theo chính sách sử dụng của MangaDex, giúp họ giám sát mạng lưới phân phối ảnh (MangaDex@Home).

Có delay nhỏ giữa các lần tải để tránh vượt rate limit.

In [ ]:
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Số ảnh tải song song cùng lúc. MangaDex cho phép ~5 request/giây,
# nên 4 luồng là mức an toàn, nhanh hơn nhiều so với tải tuần tự
# mà không bị chặn (429 Too Many Requests).
MAX_WORKERS = 4

# Hàng đợi báo cáo: các luồng tải chỉ đẩy dữ liệu vào đây,
# một luồng riêng sẽ gửi report ở nền, không làm chậm việc tải ảnh.
_report_queue = []
_report_lock = threading.Lock()

def _queue_report(page_url: str, success: bool, byte_size: int, duration_ms: int, cached: bool):
    with _report_lock:
        _report_queue.append({
            "url": page_url,
            "success": success,
            "bytes": byte_size,
            "duration": duration_ms,
            "cached": cached,
        })

def _send_single_report(item):
    try:
        requests.post("https://api.mangadex.network/report", json=item, timeout=10)
    except requests.RequestException:
        pass  # báo cáo lỗi không nên làm gián đoạn kết quả chính


def _flush_reports():
    with _report_lock:
        pending = list(_report_queue)
        _report_queue.clear()

    if not pending:
        return

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_send_single_report, item) for item in pending]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Đang gửi báo cáo"):
            pass


def _download_one(args):
    idx, filename, base_url, chapter_hash, quality_folder, output_dir, retries = args
    page_url = f"{base_url}/{quality_folder}/{chapter_hash}/{filename}"
    local_path = os.path.join(output_dir, f"{idx:03d}_{filename}")

    for attempt in range(1, retries + 1):
        try:
            start = time.time()
            r = requests.get(page_url, timeout=20)
            duration_ms = int((time.time() - start) * 1000)

            if r.status_code == 200:
                with open(local_path, "wb") as f:
                    f.write(r.content)
                cached = r.headers.get("X-Cache", "").startswith("HIT")
                _queue_report(page_url, True, len(r.content), duration_ms, cached)
                return (True, local_path, filename)
            else:
                _queue_report(page_url, False, 0, duration_ms, False)
                time.sleep(1.5 * attempt)

        except requests.RequestException:
            time.sleep(1.5 * attempt)

    return (False, None, filename)


def download_chapter_pages(base_url, chapter_hash, pages, quality_folder, output_dir, retries=3, max_workers=MAX_WORKERS):
    """Tải song song các trang của chapter bằng thread pool, có retry khi lỗi tạm thời."""
    downloaded_files = []
    failed_pages = []

    tasks = [
        (idx, filename, base_url, chapter_hash, quality_folder, output_dir, retries)
        for idx, filename in enumerate(pages, start=1)
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(_download_one, task) for task in tasks]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Đang tải trang"):
            success, local_path, filename = future.result()
            if success:
                downloaded_files.append(local_path)
            else:
                failed_pages.append(filename)

    # Sắp xếp lại theo đúng thứ tự trang gốc (thread pool trả về không theo thứ tự)
    downloaded_files.sort()

    # Gửi toàn bộ report đã gom được sau khi tải xong
    _flush_reports()

    return downloaded_files, failed_pages


start_time = time.time()
downloaded_files, failed_pages = download_chapter_pages(
    base_url, chapter_hash, pages, quality_folder, OUTPUT_DIR
)
elapsed = time.time() - start_time

print(f"\nHoàn tất: {len(downloaded_files)}/{len(pages)} trang tải thành công trong {elapsed:.1f} giây.")
if failed_pages:
    print(f"Các trang tải thất bại: {failed_pages}")
    print("Gợi ý: chạy lại cell '4. Lấy thông tin server ảnh' để lấy base URL mới rồi thử lại,")
    print("hoặc giảm MAX_WORKERS xuống 2-3 nếu nghi ngờ bị giới hạn tốc độ (rate limit).")

## 1.5 Kiểm tra nhanh kết quả

Hiển thị vài trang đầu để xác nhận ảnh tải về đúng và không bị lỗi/hỏng file.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

preview_count = min(3, len(downloaded_files))

if preview_count == 0:
    print("Không có ảnh nào để xem trước — kiểm tra lại chapter ID hoặc kết nối mạng.")
else:
    fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 8))
    if preview_count == 1:
        axes = [axes]

    for ax, path in zip(axes, downloaded_files[:preview_count]):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"\nToàn bộ {len(downloaded_files)} trang đã lưu tại:\n{OUTPUT_DIR}")

---
# BƯỚC 2: OCR — nhận diện chữ trong ảnh bằng PaddleOCR

Notebook này đọc toàn bộ ảnh đã tải ở Bước 1, chạy PaddleOCR (offline, chạy ngay trong kernel Colab, không cần API key/billing) cho từng ảnh, rồi lưu kết quả ra file JSON — mỗi ảnh một file, chứa tọa độ vùng chữ + nội dung text + điểm tin cậy (confidence). Format JSON output gồm `box`, `text_original`, `confidence`, `text_translated`, `keep` — các bước 3/4/5 phía sau dùng chung format này.

**Đặc điểm cần lưu ý:** PaddleOCR chỉ detect được từng **dòng chữ** riêng lẻ, chứ không tự gộp thành 1 bong bóng thoại hoàn chỉnh như một số dịch vụ OCR trả phí. Vì vậy nếu 1 bong bóng có nhiều dòng, mỗi dòng sẽ là 1 region riêng ở bước này — việc gộp lại thành đúng 1 bong bóng được xử lý ở Bước 4 (mục C — gộp bong bóng) dựa trên khoảng cách/độ chồng lấn giữa các dòng.

**Chi phí:** miễn phí hoàn toàn, chạy local trong Colab.

**Quy trình:**
1. Cài đặt PaddleOCR
2. Khởi tạo model + hàm chuyển đổi kết quả thành format region quen thuộc
3. Chạy cho toàn bộ ảnh trong chapter
4. Xem preview kết quả OCR


## 2.1 Cài đặt các thư viện cần thiết

`ultralytics` (chạy model YOLOv8 phát hiện bong bóng thoại) + `huggingface_hub` (tải model) + `paddleocr`/`paddlepaddle` (OCR nhận diện chữ trong từng bong bóng đã crop).

**Về xung đột thư viện:** `ultralytics` chạy thẳng ở kernel Colab chính, không cần venv riêng như PaddleOCR — vì `ultralytics` không có lịch sử xung đột với `numpy`/`scipy` của Colab (khác với PaddleOCR trước đây từng gặp vấn đề này khi chạy inpaint LaMa ở Bước 5, nên bước đó vẫn phải cô lập trong venv riêng).

In [ ]:
!pip install -q ultralytics huggingface_hub paddleocr paddlepaddle

## 2.2 Tải model YOLOv8 phát hiện bong bóng thoại

Model `comic-speech-bubble-detector.pt` (YOLOv8m, huấn luyện trên ~8k ảnh manga/webtoon/manhua) từ Hugging Face repo `ogkalu/comic-speech-bubble-detector-yolov8m`. Chỉ cần tải **1 lần** — lưu vào `PROJECT_ROOT/models` trên Drive nên các phiên Colab sau (hoặc chapter khác) dùng lại ngay, không tải lại.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
BUBBLE_MODEL_PATH = os.path.join(MODEL_DIR, 'comic-speech-bubble-detector.pt')

if not os.path.exists(BUBBLE_MODEL_PATH):
    print('Chưa có model trong Drive, đang tải từ Hugging Face...')
    downloaded_path = hf_hub_download(
        repo_id='ogkalu/comic-speech-bubble-detector-yolov8m',
        filename='comic-speech-bubble-detector.pt',
    )
    shutil.copy(downloaded_path, BUBBLE_MODEL_PATH)
    print(f'Đã tải và lưu model vào Drive: {BUBBLE_MODEL_PATH}')
else:
    print(f'Model đã có sẵn trong Drive, bỏ qua tải lại: {BUBBLE_MODEL_PATH}')

## 2.3 Khởi tạo model YOLOv8 (phát hiện bong bóng) + PaddleOCR (nhận diện chữ)

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR

bubble_detector = YOLO(BUBBLE_MODEL_PATH)

# lang='en': doi thanh 'korean', 'japan', 'ch', ... neu ban dich tu ngon ngu khac
ocr_engine = PaddleOCR(
    use_textline_orientation=True,
    use_doc_unwarping=False,
    use_doc_orientation_classify=False,
    lang='en',
    enable_mkldnn=False,  # tat oneDNN de tranh loi NotImplementedError o buoc 2.6 (bug PIR/oneDNN cua PaddlePaddle 3.3.x tren CPU)
)

print('Đã khởi tạo YOLOv8 (phát hiện bong bóng) và PaddleOCR (nhận diện chữ).')

## 2.4 Hàm phát hiện bong bóng + OCR từng bong bóng + gộp text

Quy trình cho mỗi ảnh: YOLOv8 detect bong bóng → có list box bong bóng → crop ảnh theo từng box (kèm padding nhỏ để không cắt sát chữ) → chạy PaddleOCR riêng trên từng crop → toạ độ chữ PaddleOCR trả về (theo hệ toạ độ của crop) được cộng lại offset `(x1, y1)` của box để map ngược về hệ toạ độ ảnh gốc.

Text trong cùng 1 bong bóng (dù PaddleOCR có phân tách theo dòng khi OCR trên crop) được **gộp ngay trong bước này** — vì đã biết chắc chắn toàn bộ dòng chữ trong 1 crop đều thuộc cùng 1 bong bóng, không cần đoán bằng heuristic khoảng cách/overlap như cách làm cũ ở Bước 4.

In [ ]:
import numpy as np

def detect_bubble_boxes(image_path, conf_threshold=0.25):
    """Chay YOLOv8 tren 1 anh, tra ve list box bong bong dang [x1, y1, x2, y2]
    (toa do pixel tren anh goc, chua padding). Chi giu box co confidence
    >= conf_threshold.
    """
    results = bubble_detector.predict(image_path, conf=conf_threshold, verbose=False)
    boxes = []
    for res in results:
        if res.boxes is None:
            continue
        for box_tensor in res.boxes.xyxy:
            x1, y1, x2, y2 = box_tensor.tolist()
            boxes.append([x1, y1, x2, y2])
    return boxes


def _pad_and_clip_box(box, padding, img_width, img_height):
    """Mo rong box them `padding` pixel moi phia (tranh cat sat chu khi
    crop), roi clip lai trong bien anh de khong crop ra ngoai anh goc.
    Tra ve toa do so nguyen (x1, y1, x2, y2), dung format Image.crop().
    """
    x1, y1, x2, y2 = box
    x1 = max(0, int(x1) - padding)
    y1 = max(0, int(y1) - padding)
    x2 = min(img_width, int(x2) + padding)
    y2 = min(img_height, int(y2) + padding)
    return (x1, y1, x2, y2)


def ocr_crop_to_region(image, bubble_box, min_confidence=0.0):
    """Crop 1 bong bong tu anh goc, chay PaddleOCR tren crop, gop TAT CA
    dong chu trong crop thanh 1 region duy nhat, roi map box tra ve tu
    he toa do crop ve he toa do anh goc bang cach cong lai offset (x1, y1).
    """
    x1, y1, x2, y2 = bubble_box
    crop = image.crop((x1, y1, x2, y2))
    crop_np = np.array(crop)

    result = ocr_engine.predict(crop_np)
    if not result:
        return None  # bong bong khong doc duoc chu nao

    res = result[0]
    texts = res['rec_texts']
    scores = res['rec_scores']
    polys = res['rec_polys']

    if not texts:
        return None  # bong bong khong doc duoc chu nao (VD bong bong trong/SFX ve tay)

    line_texts = []
    confidences = []
    all_points = []
    for text, confidence, box_raw in zip(texts, scores, polys):
        if not text or confidence < min_confidence:
            continue
        line_texts.append(text)
        confidences.append(confidence)
        for p in box_raw:
            all_points.append((p[0] + x1, p[1] + y1))

    if not line_texts:
        return None

    text_original = '\n'.join(line_texts)
    avg_confidence = sum(confidences) / len(confidences)

    xs = [p[0] for p in all_points]
    ys = [p[1] for p in all_points]
    merged_box = [
        [int(min(xs)), int(min(ys))],
        [int(max(xs)), int(min(ys))],
        [int(max(xs)), int(max(ys))],
        [int(min(xs)), int(max(ys))],
    ]

    return {
        'box': merged_box,
        'text_original': text_original,
        'confidence': round(float(avg_confidence), 4),
        'text_translated': '',
        'keep': True,
    }

def run_bubble_ocr_on_image(image_path, yolo_conf_threshold=0.25, ocr_min_confidence=0.0, padding=6):
    """Pipeline day du cho 1 anh: YOLOv8 detect bong bong -> crop tung
    bong bong (co padding) -> PaddleOCR moi crop -> gop cac dong chu
    trong cung 1 crop thanh 1 region. Tra ve list region o dung format
    {box, text_original, confidence, text_translated, keep}.
    """
    image = Image.open(image_path).convert('RGB')
    img_width, img_height = image.size
    bubble_boxes = detect_bubble_boxes(image_path, conf_threshold=yolo_conf_threshold)

    regions = []
    for raw_box in bubble_boxes:
        padded_box = _pad_and_clip_box(raw_box, padding, img_width, img_height)
        region = ocr_crop_to_region(image, padded_box, min_confidence=ocr_min_confidence)
        if region is not None:
            regions.append(region)
    return regions


print('Đã định nghĩa các hàm phát hiện bong bóng + OCR theo từng bong bóng.')

## 2.5 Cấu hình OCR

In [ ]:
LANG = 'en'  # phai khop voi lang da truyen vao PaddleOCR() o cell tren
MIN_CONFIDENCE = 0.5  # bo qua dong chu PaddleOCR khong chac chan (confidence thap) trong 1 bong bong
YOLO_CONF_THRESHOLD = 0.25  # nguong confidence de YOLOv8 coi la 1 bong bong hop le
BUBBLE_PADDING = 6  # so pixel mo rong quanh box bong bong truoc khi crop, tranh cat sat chu

print(f'Đọc ảnh từ: {RAW_DIR}')
print(f'Lưu kết quả OCR tại: {OCR_OUTPUT_DIR}')
print(f'Ngôn ngữ OCR: {LANG}, ngưỡng OCR confidence: {MIN_CONFIDENCE}, ngưỡng YOLO confidence: {YOLO_CONF_THRESHOLD}')

## 2.6 Chạy phát hiện bong bóng + OCR cho toàn bộ chapter

In [ ]:
import json
import time

image_files_to_ocr = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])

os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

start_time = time.time()
total_regions = 0
failed_files = []

for idx, filename in enumerate(image_files_to_ocr, start=1):
    image_path = os.path.join(RAW_DIR, filename)
    try:
        regions = run_bubble_ocr_on_image(
            image_path,
            yolo_conf_threshold=YOLO_CONF_THRESHOLD,
            ocr_min_confidence=MIN_CONFIDENCE,
            padding=BUBBLE_PADDING,
        )
    except Exception as e:
        print(f'LỖI ở trang {filename}: {e}')
        failed_files.append(filename)
        continue

    total_regions += len(regions)

    json_filename = os.path.splitext(filename)[0] + '.json'
    json_path = os.path.join(OCR_OUTPUT_DIR, json_filename)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump({'source_image': filename, 'regions': regions}, f, ensure_ascii=False, indent=2)

    print(f'PROGRESS {idx}/{len(image_files_to_ocr)} {filename} bong_bong={len(regions)}')

elapsed = time.time() - start_time
print(f'\nDONE total_images={len(image_files_to_ocr)} total_regions={total_regions}')
print(f'Hoàn tất trong {elapsed:.1f} giây.')
if failed_files:
    print(f'\nCẢNH BÁO: {len(failed_files)} trang lỗi, chưa có kết quả OCR: {failed_files}')
    print('Có thể chạy lại cell này để thử lại toàn bộ (tốn thêm thời gian xử lý cho các trang đã thành công).')

## 2.7 Xem trước kết quả OCR trên ảnh

Vẽ khung đỏ quanh mỗi vùng chữ (đã gộp theo bong bóng) lên ảnh, kèm chỉ số confidence.

In [ ]:
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

image_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])

def draw_ocr_preview(image_path: str, regions: list):
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    for region in regions:
        box = region['box']
        points = [(p[0], p[1]) for p in box]
        draw.polygon(points, outline='red', width=3)
        x, y = points[0]
        draw.text((x, max(0, y - 15)), f"{region['confidence']:.2f}", fill='red')
    return img


PREVIEW_COUNT = 3
preview_files = image_files[:PREVIEW_COUNT]
fig, axes = plt.subplots(1, len(preview_files), figsize=(7 * len(preview_files), 10))
if len(preview_files) == 1:
    axes = [axes]

for ax, filename in zip(axes, preview_files):
    json_path = os.path.join(OCR_OUTPUT_DIR, os.path.splitext(filename)[0] + '.json')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    preview_img = draw_ocr_preview(os.path.join(RAW_DIR, filename), data['regions'])
    ax.imshow(preview_img)
    ax.set_title(f"{filename} ({len(data['regions'])} vùng chữ)", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2.8 Xem nhanh nội dung text đã OCR (dạng danh sách)

In [ ]:
CHECK_PAGE_INDEX = 0

check_filename = image_files[CHECK_PAGE_INDEX]
check_json_path = os.path.join(OCR_OUTPUT_DIR, os.path.splitext(check_filename)[0] + '.json')

with open(check_json_path, 'r', encoding='utf-8') as f:
    check_data = json.load(f)

print(f"Trang: {check_data['source_image']}")
print(f"Số vùng chữ: {len(check_data['regions'])}\n")

for i, region in enumerate(check_data['regions'], start=1):
    print(f"[{i}] (conf={region['confidence']:.2f}) {region['text_original']}")

---
# BƯỚC 3: Dịch sang tiếng Việt bằng Groq API (LLM, miễn phí, không cần thẻ tín dụng)

Notebook này đọc kết quả OCR từ Bước 2 (`text_original`), dịch sang tiếng Việt bằng **Groq API** (chạy model LLM mã nguồn mở `openai/gpt-oss-120b` trên hạ tầng Groq), rồi điền vào trường `text_translated` trong các file JSON.

**Vì sao dùng Groq thay vì Argos Translate (LibreTranslate) hay DeepL/Google:**

- DeepL và Google Cloud Translation đều yêu cầu thẻ tín dụng quốc tế để kích hoạt kể cả gói miễn phí — nhiều thẻ Việt Nam bị từ chối ở bước xác minh.
- Argos Translate (dùng trước đây) là dịch máy truyền thống, chất lượng thấp — và **quan trọng hơn: không có model dịch trực tiếp Trung/Nhật/Hàn → Việt**, chỉ có Anh ↔ X, nên dịch các ngôn ngữ khác phải qua 2 chặng (VD Hàn→Anh→Việt), làm chất lượng giảm thêm.
- Groq cung cấp quyền truy cập miễn phí (không thẻ tín dụng) vào các model LLM mã nguồn mở tốc độ cao, có thể dịch **trực tiếp** từ bất kỳ ngôn ngữ nguồn nào sang tiếng Việt với chất lượng tự nhiên hơn hẳn dịch máy truyền thống — LLM hiểu ngữ cảnh, giữ được văn phong hội thoại.

**Cách lấy API key (miễn phí, không cần thẻ):**
1. Vào https://console.groq.com , đăng ký bằng email/Google
2. Vào mục API Keys, tạo key mới
3. **KHÔNG dán key trực tiếp vào notebook hay chat với AI bất kỳ** — vào Colab, bấm biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm secret tên `GROQ_API_KEY`, dán key vào đó, bật toggle "Notebook access"

**Giới hạn free tier (tính đến 7/2026):** 30 request/phút, 1.000 request/ngày — notebook gộp toàn bộ vùng chữ của 1 trang thành 1 request duy nhất, nên 1 chapter ~50-60 trang chỉ tốn ~50-60 request, nằm rất thoải mái trong giới hạn.

**Lưu ý về model:** `llama-3.3-70b-versatile` (từng dùng phổ biến) đã bị Groq thông báo deprecate (17/6/2026) và dự kiến tắt hẳn vào 8/2026, nên notebook này dùng `openai/gpt-oss-120b` — model được Groq khuyến nghị thay thế chính thức.

## 3.1 Cài Groq SDK + đọc API key từ Colab Secrets

In [ ]:
!pip install -q groq

from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    raise RuntimeError(
        'Không tìm thấy GROQ_API_KEY trong Colab Secrets. '
        'Bấm biểu tượng 🔑 ở thanh bên trái Colab, thêm secret tên GROQ_API_KEY '
        'với giá trị là API key lấy từ https://console.groq.com/keys, '
        'bật "Notebook access", rồi chạy lại cell này.'
    )

print('Đã đọc GROQ_API_KEY từ Colab Secrets thành công.')

## 3.2 Cấu hình dịch

`FROM_LANG` chỉ dùng để mô tả ngôn ngữ nguồn trong prompt gửi cho model (không giới hạn theo mã ISO cụ thể như Argos) — có thể ghi rõ bằng tên ngôn ngữ tiếng Anh, VD `"Korean"`, `"Japanese"`, `"Chinese"`, `"English"`.

In [ ]:
FROM_LANG = "English"  # Ngôn ngữ nguồn trong ảnh — đổi thành "Korean", "Japanese", "Chinese" v.v. tùy truyện
TO_LANG = "Vietnamese"

GROQ_MODEL = "openai/gpt-oss-120b"

# Mặc định False: chỉ dịch những vùng còn trống text_translated, giúp
# chạy lại an toàn mà không ghi đè các chỗ đã sửa thủ công.
OVERWRITE_EXISTING = False

print(f'Đọc/ghi kết quả tại: {OCR_OUTPUT_DIR}')
print(f'Dịch: {FROM_LANG} → {TO_LANG} (model: {GROQ_MODEL})')
print(f'Ghi đè bản dịch cũ: {OVERWRITE_EXISTING}')

## 3.3 Các hàm gọi Groq API

Gộp toàn bộ vùng chữ cần dịch của 1 trang thành **1 request duy nhất** — vừa tiết kiệm quota (30 request/phút), vừa cho model ngữ cảnh tốt hơn (dịch cả trang cùng lúc giữ mạch truyện nhất quán hơn dịch từng câu rời rạc).

Có retry tự động khi gặp lỗi 429 (rate limit) — đợi theo thời gian Groq đề xuất trong response rồi thử lại.

In [ ]:
import json
import time
import re
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)


def build_translate_prompt(texts, from_lang, to_lang):

    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(texts))
    return f"""You are translating dialogue from a manhwa/manga comic from {from_lang} to {to_lang}.

Translate each numbered line below into natural, colloquial {to_lang} suitable for comic speech bubbles. Keep the tone casual and conversational, matching how people actually speak. If a line's relationship context (pronouns like "tôi/tao/mình" vs "anh/em/cậu") is unclear from a single isolated line, prefer omitting the pronoun rather than guessing — a human editor will review and adjust afterward.

Lines to translate:
{numbered}

Respond with ONLY a JSON object in this exact format, no other text:
{{"translations": ["translation 1", "translation 2", ...]}}

The "translations" array must have EXACTLY {len(texts)} elements, in the same order as the input lines."""


def call_groq_translate(texts, from_lang, to_lang, model, max_retries=8):

    if not texts:
        return [], True

    prompt = build_translate_prompt(texts, from_lang, to_lang)

    for attempt in range(1, max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.3,
            )
            content = response.choices[0].message.content
            parsed = json.loads(content)
            translations = parsed.get("translations", [])

            if len(translations) != len(texts):
                print(f"  CẢNH BÁO: model trả về {len(translations)} bản dịch, kỳ vọng {len(texts)} (lần {attempt}/{max_retries}).")
                if attempt == max_retries:
                    return [], False
                time.sleep(2)
                continue

            return translations, True

        except Exception as e:
            error_str = str(e)
            is_rate_limit = "429" in error_str or "rate_limit" in error_str.lower()
            is_auth_error = "401" in error_str or "invalid_api_key" in error_str.lower()

            if is_rate_limit:
                # Co gang doc so giay can doi tu thong bao loi (Groq thuong
                # tra ve "Please try again in Xs" trong noi dung loi)
                wait_match = re.search(r"try again in (\d+\.?\d*)s", error_str)
                wait_time = float(wait_match.group(1)) + 1 if wait_match else 5 * attempt
                print(f"  Rate limit (lần {attempt}/{max_retries}), đợi {wait_time:.0f}s...")
                time.sleep(wait_time)
            elif is_auth_error:
                # 401 quan sat thay co the tam thoi (xem docstring), doi
                # lau hon rate limit thong thuong truoc khi thu lai
                wait_time = min(10 * attempt, 60)
                print(f"  Lỗi xác thực 401 (lần {attempt}/{max_retries}, có thể tạm thời phía Groq), đợi {wait_time}s...")
                time.sleep(wait_time)
            elif attempt < max_retries:
                print(f"  Lỗi gọi Groq (lần {attempt}/{max_retries}): {e}. Thử lại sau 3s...")
                time.sleep(3)
            else:
                print(f"  Lỗi gọi Groq sau {max_retries} lần thử: {e}")
                return [], False

    return [], False


print("Đã định nghĩa xong các hàm gọi Groq API.")

## 3.3.5 Kiểm tra bản dịch rỗng còn sót từ lần chạy trước (nếu có)

Nếu bạn từng chạy notebook bằng phiên bản code cũ (trước khi sửa lỗi không âm thầm ghi chuỗi rỗng), có thể một số vùng có `text_original` khác rỗng nhưng `text_translated` bị ghi đè thành chuỗi rỗng do lỗi tạm thời của Groq trước đây. Cell này chỉ **quét và báo cáo**, không tự sửa gì — các vùng này sẽ tự động được dịch lại ở mục 3.4 bên dưới vì `text_translated` rỗng được coi là "chưa dịch".

In [ ]:
json_files_check = sorted([f for f in os.listdir(OCR_OUTPUT_DIR) if f.lower().endswith('.json')])

empty_translation_count = 0
for filename in json_files_check:
    path = os.path.join(OCR_OUTPUT_DIR, filename)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for region in data.get('regions', []):
        if not region.get('keep', True):
            continue
        has_original = bool(region.get('text_original', '').strip())
        has_translation = bool(region.get('text_translated', '').strip())
        if has_original and not has_translation:
            empty_translation_count += 1

if empty_translation_count > 0:
    print(f'Tìm thấy {empty_translation_count} vùng có text gốc nhưng bản dịch đang RỖNG.')
    print('Đây có thể là dấu vết từ lần chạy trước (nếu dùng code cũ). Không cần làm gì —')
    print('mục 3.4 bên dưới sẽ tự động dịch lại các vùng này vì OVERWRITE_EXISTING=False')
    print('chỉ bỏ qua vùng đã CÓ bản dịch, còn vùng rỗng vẫn được coi là "chưa dịch".')
else:
    print('Không có vùng nào bị rỗng bất thường. Sẵn sàng chạy mục 3.4.')

In [ ]:
from google.colab import userdata
from groq import Groq

key = userdata.get('GROQ_API_KEY')
print("Độ dài key:", len(key) if key else None)
print("Đầu key:", repr(key[:8]) if key else None)
print("Cuối key:", repr(key[-8:]) if key else None)

try:
    test_client = Groq(api_key=key)
    resp = test_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": "ping"}],
    )
    print("THÀNH CÔNG:", resp.choices[0].message.content)
except Exception as e:
    print("LỖI ĐẦY ĐỦ:", repr(e))

try:
    test_client2 = Groq(api_key=key)
    resp2 = test_client2.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": 'Reply with JSON: {"msg": "pong"}'}],
        response_format={"type": "json_object"},
    )
    print("THÀNH CÔNG:", resp2.choices[0].message.content)
except Exception as e:
    print("LỖI ĐẦY ĐỦ:", repr(e))

## 3.4 Chạy dịch toàn bộ chapter

Duyệt từng trang, gộp các vùng cần dịch thành 1 request/trang, có delay nhỏ giữa các trang để tôn trọng giới hạn 30 request/phút của Groq free tier.

In [ ]:
json_files = sorted([f for f in os.listdir(OCR_OUTPUT_DIR) if f.lower().endswith('.json')])
print(f'Tìm thấy {len(json_files)} file JSON cần dịch.\n')

# Khoang cach toi thieu giua cac request, de khong vuot 30 request/phut
# (60s / 30 = 2s/request, dat du 2.5s de co du phong).
MIN_SECONDS_BETWEEN_REQUESTS = 2.5

total_regions_translated = 0
total_regions_skipped = 0
failed_pages = []  # ten cac file THUC SU that bai sau toan bo retry -- can dich lai
last_request_time = 0

start_time = time.time()

for idx, filename in enumerate(json_files, start=1):
    json_path = os.path.join(OCR_OUTPUT_DIR, filename)
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    regions = data.get('regions', [])

    # Chi dich vung keep=True va (chua co ban dich HOAC dang ghi de)
    regions_to_translate = []
    indices_to_translate = []
    for i, region in enumerate(regions):
        if not region.get('keep', True):
            continue
        already_translated = bool(region.get('text_translated', '').strip())
        if already_translated and not OVERWRITE_EXISTING:
            total_regions_skipped += 1
            continue
        regions_to_translate.append(region.get('text_original', ''))
        indices_to_translate.append(i)

    if not regions_to_translate:
        print(f'[{idx}/{len(json_files)}] {filename}: không có vùng cần dịch, bỏ qua.')
        continue

    # Ton trong khoang cach toi thieu giua cac request
    elapsed_since_last = time.time() - last_request_time
    if elapsed_since_last < MIN_SECONDS_BETWEEN_REQUESTS:
        time.sleep(MIN_SECONDS_BETWEEN_REQUESTS - elapsed_since_last)

    translations, success = call_groq_translate(regions_to_translate, FROM_LANG, TO_LANG, GROQ_MODEL)
    last_request_time = time.time()

    if not success:
        # KHONG ghi gi vao JSON ca -- de nguyen text_translated cu (co the
        # rong) de vong lap sau (hoac chay lai cell nay) con nhan ra day
        # la vung chua dich va thu lai, thay vi bi che lap boi chuoi rong.
        failed_pages.append(filename)
        print(f'[{idx}/{len(json_files)}] {filename}: THẤT BẠI sau nhiều lần thử, bỏ qua trang này (sẽ cần dịch lại).')
        continue

    for region_idx, translated_text in zip(indices_to_translate, translations):
        regions[region_idx]['text_translated'] = translated_text
        total_regions_translated += 1

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f'[{idx}/{len(json_files)}] {filename}: đã dịch {len(regions_to_translate)} vùng.')

elapsed = time.time() - start_time
print(f'\nHoàn tất dịch trong {elapsed:.1f} giây.')
print(f'Tổng số vùng đã dịch: {total_regions_translated}, bỏ qua (đã có bản dịch): {total_regions_skipped}.')

if failed_pages:
    print(f'\n{"="*60}')
    print(f'CẢNH BÁO: {len(failed_pages)} TRANG DỊCH THẤT BẠI, CẦN CHẠY LẠI:')
    print(f'{"="*60}')
    for fp in failed_pages:
        print(f'  - {fp}')
    print('\nChạy lại CHÍNH cell này (mục 3.4) một lần nữa — vì OVERWRITE_EXISTING=False,')
    print('nó sẽ tự động BỎ QUA các trang đã dịch thành công và CHỈ thử lại các trang lỗi ở trên.')
else:
    print('\nMọi trang đều dịch thành công, không có trang nào cần chạy lại.')

## 3.5 Xem trước kết quả dịch

In [ ]:
CHECK_PAGE_INDEX = 1  # đổi số này để xem trang khác (khớp preview Bước 2, trang có nhiều thoại)

check_filename = json_files[CHECK_PAGE_INDEX]
check_json_path = os.path.join(OCR_OUTPUT_DIR, check_filename)

with open(check_json_path, 'r', encoding='utf-8') as f:
    check_data = json.load(f)

print(f"Trang: {check_data['source_image']}")
print(f"Số vùng chữ: {len(check_data['regions'])}\n")

for i, region in enumerate(check_data['regions'], start=1):
    print(f"[{i}] Gốc:  {region['text_original']}")
    print(f"    Dịch: {region['text_translated']}")
    print()

---
# BƯỚC 4: Làm sạch vùng chữ (lọc tên chương/logo)

Notebook này xử lý JSON kết quả OCR trước khi inpaint/chèn chữ, gồm 2 việc:

**A. Tự động bỏ qua tên chương dạng `#12_...`**

Tên chapter (thường nằm trong ô giấy dán ở đầu trang, dạng `#12_INSIDE A TAXI`) bị OCR nhận diện như text thường, dẫn tới bị xóa nhầm ở Bước 5 (inpaint) dù đây là logo/tiêu đề cần giữ nguyên, không phải thoại. Notebook tự động nhận diện theo pattern (bắt đầu bằng `#` + số) và đặt `keep=False`.

**B. Cảnh báo (không tự sửa) các vùng nghi là logo thương hiệu**

Logo tên truyện (chữ Hàn/font cách điệu, VD "테토X메겐") có thể bị OCR đọc sai lung tung mỗi lần một kiểu khác nhau — không có pattern text cố định để tự động lọc an toàn. Notebook chỉ **in cảnh báo** cho các vùng có dấu hiệu nghi ngờ (confidence thấp bất thường + nằm ở phần đầu trang), bạn tự quyết định tắt `keep` cho vùng nào ở Bước 6 (Gradio) — tránh nguy cơ tự động xóa nhầm thoại thật.

**Không còn bước gộp bong bóng ở đây:** vì Bước 2 (YOLOv8 + PaddleOCR theo từng bong bóng) đã gộp toàn bộ dòng chữ trong cùng 1 bong bóng thành 1 region ngay từ lúc OCR — không cần đoán lại bằng heuristic khoảng cách/overlap như hồi còn OCR nguyên trang.

**Chạy notebook này ở đâu trong pipeline:** sau Bước 3 (dịch) — vì lọc tên chương dựa trên `text_original`, không phụ thuộc bản dịch — và trước Bước 5 (inpaint)/Bước 6 (Gradio).

**An toàn dữ liệu:** trước khi ghi đè, mỗi file JSON gốc được sao lưu vào thư mục `ocr_results_backup_before_merge/` — nếu lọc sai chỗ nào, có thể khôi phục lại từ đó.

## 4.1 Cấu hình lọc

In [ ]:
# --- Tham số lọc tên chương ---
CHAPTER_TITLE_PATTERN = r'^#\s*\d+'

# --- Tham số cảnh báo logo ---
LOGO_WARN_MAX_CONFIDENCE = 0.75
LOGO_WARN_TOP_PAGE_RATIO = 0.35

print(f'Đọc/ghi JSON tại: {OCR_OUTPUT_DIR}')
print(f'Backup JSON gốc tại: {BACKUP_DIR}')

## 4.2 Các hàm lọc

In [ ]:
import re


def is_chapter_title(text, pattern):
    """Nhan dien text dang tieu de chapter (VD '#12_INSIDE A TAXI').
    Dung regex match tu dau chuoi (sau khi loai khoang trang thua)."""
    return bool(re.match(pattern, text.strip()))


def filter_chapter_titles(regions, pattern):
    """Duyet toan bo region trong 1 trang, tu dong dat keep=False cho
    vung khop pattern tieu de chapter. Tra ve (regions_da_sua, so_luong_sua)."""
    count = 0
    for region in regions:
        if region.get('keep', True) and is_chapter_title(region.get('text_original', ''), pattern):
            region['keep'] = False
            region['auto_filtered_reason'] = 'chapter_title_pattern'
            count += 1
    return regions, count


def box_bounds(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    return min(xs), min(ys), max(xs), max(ys)


def find_logo_warnings(regions, page_height, max_confidence, top_ratio):
    """Tim cac vung NGHI NGO la logo/ten truyen/ten tac gia, dua tren 2
    dau hieu: confidence OCR thap bat thuong (font cach dieu/chu Han
    thuong khien OCR doc sai -> confidence thap) VA nam o phan tren cung
    trang. CHI tra ve danh sach de canh bao, KHONG tu dong doi keep --
    vi khong co pattern text on dinh de loc an toan."""
    warnings = []
    for region in regions:
        if not region.get('keep', True):
            continue
        box = region.get('box')
        confidence = region.get('confidence', 1.0)
        if not box or page_height <= 0:
            continue
        _, y0, _, _ = box_bounds(box)
        is_near_top = (y0 / page_height) <= top_ratio
        is_low_confidence = confidence <= max_confidence
        if is_near_top and is_low_confidence:
            warnings.append(region)
    return warnings


print('Đã định nghĩa xong các hàm lọc.')

## 4.3 Xem trước thay đổi (không ghi file)

In ra 2 phần cho từng trang có thay đổi:
- **Tên chương tự động lọc** (mục A) — sẽ tự đặt `keep=False`
- **Cảnh báo nghi ngờ logo** (mục B) — CHỈ liệt kê, không tự đổi gì, bạn tự kiểm tra và tắt tay ở Bước 6 nếu đúng là logo

RAW_DIR cần có sẵn (từ Bước 1) để đọc kích thước ảnh thật, phục vụ tính vị trí tương đối cho phần cảnh báo logo.

In [ ]:
import json
from PIL import Image

RAW_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'raw')

json_files = sorted([f for f in os.listdir(OCR_OUTPUT_DIR) if f.lower().endswith('.json')])
print(f'Tìm thấy {len(json_files)} file JSON.\n')


def get_page_height(filename):
    """Doc chieu cao thuc te cua anh goc, dung de tinh vi tri tuong doi
    cho phan canh bao logo. Neu khong tim thay anh (VD chi co JSON, chua
    tai anh), fallback dung y lon nhat trong cac box lam uoc luong."""
    base_name = os.path.splitext(filename)[0]
    for ext in ('.jpg', '.jpeg', '.png', '.webp'):
        candidate = os.path.join(RAW_DIR, base_name + ext)
        if os.path.exists(candidate):
            with Image.open(candidate) as img:
                return img.height
    return None


total_title_filtered_preview = 0
total_logo_warnings_preview = 0

for filename in json_files:
    path = os.path.join(OCR_OUTPUT_DIR, filename)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    regions = data.get('regions', [])
    page_height = get_page_height(filename)
    if page_height is None:
        # Fallback: uoc luong bang y lon nhat xuat hien trong cac box
        page_height = max((box_bounds(r['box'])[3] for r in regions), default=1)

    # --- A: tim (khong sua) tieu de chapter se bi loc ---
    title_hits = [r for r in regions if r.get('keep', True) and is_chapter_title(r.get('text_original', ''), CHAPTER_TITLE_PATTERN)]

    # --- B: canh bao logo (chi liet ke) ---
    logo_warnings = find_logo_warnings(regions, page_height, LOGO_WARN_MAX_CONFIDENCE, LOGO_WARN_TOP_PAGE_RATIO)
    # loai nhung vung da nam trong title_hits de khong bao trung
    logo_warnings = [r for r in logo_warnings if r not in title_hits]

    if title_hits or logo_warnings:
        print(f'{filename}:')
        for r in title_hits:
            print(f'  [A] Tự động lọc tên chương: \"{r.get("text_original", "")}\"')
        for r in logo_warnings:
            print(f'  [B] CẢNH BÁO nghi logo (conf={r.get("confidence", 0):.2f}): \"{r.get("text_original", "")}\" -> tự kiểm tra ở Bước 6')

    total_title_filtered_preview += len(title_hits)
    total_logo_warnings_preview += len(logo_warnings)

print(f'\nTổng dự kiến: {total_title_filtered_preview} tên chương tự lọc, '
      f'{total_logo_warnings_preview} vùng cảnh báo logo.')
print('Nếu thấy hợp lý, chạy tiếp mục 4.4 để ghi đè thật.')

## 4.4 Thực hiện lọc và ghi đè JSON

Thứ tự xử lý cho mỗi trang: **(A) lọc tên chương** → **(B) thu thập cảnh báo logo** (không sửa).

Mỗi file JSON gốc được sao lưu vào `BACKUP_DIR` trước khi ghi đè (chỉ backup lần đầu — nếu file backup đã tồn tại từ lần chạy trước, giữ nguyên bản backup cũ, không ghi đè backup, để tránh vô tình backup luôn bản đã lọc lỗi).

Cuối cùng in ra **toàn bộ danh sách cảnh báo logo** gộp từ mọi trang — bạn dùng danh sách này để lên Bước 6 (Gradio) kiểm tra và tắt `keep` tay cho đúng vùng logo/tác giả nếu xác nhận đúng.

In [ ]:
import shutil

total_regions_before = 0
total_regions_after = 0
total_pages_changed = 0
total_title_filtered = 0
all_logo_warnings = []  # (filename, region) — tong hop de in cuoi cung

for filename in json_files:
    path = os.path.join(OCR_OUTPUT_DIR, filename)
    backup_path = os.path.join(BACKUP_DIR, filename)

    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    original_regions = data.get('regions', [])
    original_count = len(original_regions)
    page_height = get_page_height(filename)
    if page_height is None:
        page_height = max((box_bounds(r['box'])[3] for r in original_regions), default=1)

    # A: loc ten chuong (sua truc tiep keep tren list, IN-PLACE)
    working_regions, n_filtered = filter_chapter_titles(original_regions, CHAPTER_TITLE_PATTERN)
    total_title_filtered += n_filtered

    # B: thu thap canh bao logo (SAU khi da loc ten chuong, de khong bao trung)
    warnings = find_logo_warnings(working_regions, page_height, LOGO_WARN_MAX_CONFIDENCE, LOGO_WARN_TOP_PAGE_RATIO)
    for w in warnings:
        all_logo_warnings.append((filename, w))

    total_regions_before += original_count
    total_regions_after += len(working_regions)

    changed = n_filtered > 0
    if changed:
        if not os.path.exists(backup_path):
            shutil.copy2(path, backup_path)

        data['regions'] = working_regions
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        total_pages_changed += 1
        print(f'{filename}: lọc {n_filtered} tên chương')

print(f'\nHoàn tất. {total_pages_changed}/{len(json_files)} trang có thay đổi.')
print(f'Tổng số vùng: {total_regions_before} -> {total_regions_after}.')
print(f'Tổng tên chương tự động lọc: {total_title_filtered}.')
print(f'Bản gốc trước khi lọc đã lưu tại: {BACKUP_DIR}')

print(f'\n{"="*60}')
print(f'DANH SÁCH CẢNH BÁO LOGO/TÁC GIẢ NGHI NGỜ ({len(all_logo_warnings)} vùng)')
print('Tự kiểm tra và tắt keep tay ở Bước 6 (Gradio) nếu đúng là logo:')
print(f'{"="*60}')
for filename, region in all_logo_warnings:
    print(f'  {filename}: conf={region.get("confidence", 0):.2f} text="{region.get("text_original", "")}"')

## 4.5 Kiểm tra nhanh 1 trang sau khi lọc

In [ ]:
CHECK_PAGE_INDEX = 0  # đổi số này để xem trang khác

check_filename = json_files[CHECK_PAGE_INDEX]
check_path = os.path.join(OCR_OUTPUT_DIR, check_filename)

with open(check_path, 'r', encoding='utf-8') as f:
    check_data = json.load(f)

print(f"Trang: {check_data['source_image']}")
print(f"Số vùng chữ: {len(check_data['regions'])}\n")

for i, region in enumerate(check_data['regions'], start=1):
    tags = []
    if region.get('auto_filtered_reason'):
        tags.append(f"TỰ LỌC: {region['auto_filtered_reason']}")
    tag_str = f" [{', '.join(tags)}]" if tags else ""
    print(f"[{i}]{tag_str} keep={region.get('keep')} conf={region.get('confidence', 0):.2f}")
    print(f"    Gốc:  {region.get('text_original', '')}")
    print(f"    Dịch: {region.get('text_translated', '')}")
    print()

---
# BƯỚC 5: Xóa chữ trong ảnh bằng LaMa Inpainting

Notebook này đọc kết quả OCR từ Bước 2 (tọa độ `box` của từng vùng chữ), dựng mask cho các vùng **thoại** (`keep == True`), rồi dùng LaMa để "xóa" chữ và vẽ lại nền một cách tự nhiên. Ảnh kết quả lưu riêng, không ghi đè ảnh gốc.

**SFX xử lý ra sao:** các vùng đánh dấu `keep == False` (thường là SFX) sẽ **không bị đưa vào mask** — tức là giữ nguyên trên ảnh, không xóa. Nếu sau này bạn muốn xóa cả SFX (ví dụ để vẽ SFX tiếng Việt đè lên), có thể đổi `keep` của vùng đó thành `True` trước khi chạy notebook này, hoặc bật tùy chọn `INCLUDE_SFX` ở mục cấu hình.

**Vì sao cần mask riêng thay vì xóa nguyên vùng box:** Box OCR thường ôm khá sát chữ, nếu xóa đúng y hệt vùng đó thì viền chữ mờ (anti-alias) hay bị sót lại. Mask được vẽ rộng hơn box một chút (dilate) để đảm bảo xóa sạch.

**Về kiến trúc cô lập:** giống hệt Bước 2 (OCR) và Bước 3 (dịch) — cài `simple-lama-inpainting` (+ `torch`) trong một virtual environment riêng, chạy qua subprocess. Lý do: LaMa cần `torch`, có thể xung đột với các bản `torch`/`numpy` đã có sẵn trên Colab (đã từng gặp vấn đề tương tự với PaddleOCR ở Bước 2), nên cách ly cho chắc.

**Quy trình:**
1. Kết nối Google Drive
2. Tạo venv riêng, cài `simple-lama-inpainting` bên trong
3. Viết script `run_inpaint.py` chạy trong venv (dựng mask từ JSON OCR + gọi LaMa)
4. Cấu hình
5. Chạy inpaint qua subprocess
6. Xem trước kết quả (so sánh ảnh gốc / mask / ảnh đã xóa chữ)

## 5.1 Tạo virtual environment riêng cho LaMa

Giống Bước 2/3, venv nằm ở `/content/` (ổ tạm, không phải Drive) nên sẽ mất khi phiên Colab kết thúc — cần chạy lại mỗi khi mở phiên mới. Bước này tải `torch` (khá nặng, ~vài trăm MB) nên có thể mất vài phút.

In [ ]:
VENV_DIR = '/content/lama_venv'

# Cài gói hệ thống python3-venv trước để đảm bảo ensurepip hoạt động đúng
# (rút kinh nghiệm từ lỗi 'ensurepip ... returned non-zero exit status 1'
# đã gặp ở Bước 2).
!apt-get install -y -qq python3-venv > /dev/null 2>&1

!rm -rf {VENV_DIR}
!python3 -m venv {VENV_DIR}

VENV_PY = f'{VENV_DIR}/bin/python'
VENV_PIP = f'{VENV_DIR}/bin/pip'

import os as _os

# Fallback nếu venv vẫn thiếu pip (giống Bước 2/3)
if not _os.path.exists(VENV_PIP):
    print('venv không có sẵn pip, đang tự cài qua get-pip.py...')
    !curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
    !{VENV_PY} /tmp/get-pip.py

if not _os.path.exists(VENV_PIP):
    raise RuntimeError(
        'Không thể cài pip vào venv bằng cả hai cách (ensurepip và get-pip.py). '
        'Xem log phía trên để biết chi tiết lỗi.'
    )

!{VENV_PIP} install -q --upgrade pip

# simple-lama-inpainting: wrapper gọn nhẹ quanh model LaMa gốc, tự tải
# sẵn checkpoint đã pretrain (~200MB) trong lần chạy đầu tiên, không cần
# tự train hay tải checkpoint thủ công.
!{VENV_PIP} install -q simple-lama-inpainting opencv-python-headless numpy pillow

# Kiểm tra thực sự đã cài thành công trước khi đi tiếp
check = !{VENV_PY} -c "from simple_lama_inpainting import SimpleLama; print('simple_lama OK')"
print('\n'.join(check))
if not any('simple_lama OK' in line for line in check):
    raise RuntimeError(
        'Cài đặt simple-lama-inpainting trong venv KHÔNG thành công. '
        'Xem log pip install phía trên để tìm nguyên nhân cụ thể trước khi chạy tiếp.'
    )

print('Đã tạo venv và cài xong simple-lama-inpainting (cô lập hoàn toàn khỏi Python hệ thống Colab).')

## 5.2 Viết script inpaint độc lập (`run_inpaint.py`)

Script này chạy bằng Python của venv. Với mỗi ảnh: đọc JSON OCR tương ứng, dựng mask từ `box` của các vùng có `keep == True`, giãn mask (dilate) để xóa sạch viền chữ, rồi gọi LaMa để inpaint. Ảnh kết quả ghi ra thư mục output, mask cũng được lưu lại để tiện kiểm tra/debug.

In [ ]:
inpaint_script = '''
import os
import sys
import json
import argparse

import numpy as np
import cv2
from PIL import Image

from simple_lama_inpainting import SimpleLama


def build_mask(image_size, regions, include_sfx, dilate_px):
    """Dung mask trang/den tu danh sach region cua 1 trang.

    Vung TRANG (255) = se bi LaMa xoa va ve lai.
    Vung DEN (0) = giu nguyen.

    Mac dinh chi dua vao mask cac region co keep == True (thoai).
    Neu include_sfx=True thi lay toan bo region bat ke keep, phong khi
    nguoi dung muon xoa ca SFX de ve SFX tieng Viet de len.
    """
    width, height = image_size
    mask = np.zeros((height, width), dtype=np.uint8)

    for region in regions:
        if not include_sfx and not region.get("keep", True):
            continue  # bo qua SFX / vung da danh dau khong dich -> giu nguyen anh

        box = region.get("box")
        if not box:
            continue

        points = np.array([[int(p[0]), int(p[1])] for p in box], dtype=np.int32)
        cv2.fillPoly(mask, [points], 255)

    if dilate_px > 0:
        # Gian mask ra vai pixel de xoa sach vien chu mo (anti-alias) con
        # sot lai neu chi xoa dung khit vung box OCR tra ve.
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_px * 2 + 1, dilate_px * 2 + 1))
        mask = cv2.dilate(mask, kernel, iterations=1)

    return mask


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw_dir", required=True, help="Thu muc chua anh goc")
    parser.add_argument("--ocr_dir", required=True, help="Thu muc chua file JSON ket qua OCR")
    parser.add_argument("--output_dir", required=True, help="Thu muc luu anh da xoa chu")
    parser.add_argument("--mask_dir", default="", help="Thu muc luu mask de debug (bo trong = khong luu)")
    parser.add_argument("--include_sfx", action="store_true", help="Xoa ca vung keep=False (SFX)")
    parser.add_argument("--dilate_px", type=int, default=3, help="So pixel gian mask them quanh box")
    parser.add_argument("--limit", type=int, default=0, help="Chi xu ly N anh dau tien (0 = toan bo)")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    if args.mask_dir:
        os.makedirs(args.mask_dir, exist_ok=True)

    print("Dang tai model LaMa (lan dau co the mat vai phut de tai checkpoint)...", flush=True)
    simple_lama = SimpleLama()
    print("Da tai xong model LaMa.", flush=True)

    image_files = sorted([
        f for f in os.listdir(args.raw_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ])

    if args.limit > 0:
        image_files = image_files[:args.limit]

    total_regions_masked = 0
    total_skipped_no_json = 0

    for idx, filename in enumerate(image_files, start=1):
        image_path = os.path.join(args.raw_dir, filename)
        json_path = os.path.join(args.ocr_dir, os.path.splitext(filename)[0] + ".json")

        if not os.path.exists(json_path):
            print(f"CANH BAO: khong tim thay JSON OCR cho {filename}, bo qua (giu anh goc).", flush=True)
            total_skipped_no_json += 1
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            ocr_data = json.load(f)
        regions = ocr_data.get("regions", [])

        img = Image.open(image_path).convert("RGB")
        mask_arr = build_mask(img.size, regions, args.include_sfx, args.dilate_px)
        has_mask_content = mask_arr.max() > 0

        if has_mask_content:
            mask_img = Image.fromarray(mask_arr).convert("L")
            result_img = simple_lama(img, mask_img)
            # simple-lama-inpainting co the tra ve numpy array hoac PIL Image
            # tuy phien ban; chuan hoa ve PIL Image truoc khi luu.
            if not isinstance(result_img, Image.Image):
                result_img = Image.fromarray(np.array(result_img).astype(np.uint8))
        else:
            # Khong co vung nao can xoa (VD trang khong co thoai) -> giu nguyen anh goc
            result_img = img
            mask_img = Image.fromarray(mask_arr).convert("L")

        output_path = os.path.join(args.output_dir, filename)
        result_img.save(output_path)

        if args.mask_dir:
            mask_path = os.path.join(args.mask_dir, os.path.splitext(filename)[0] + "_mask.png")
            mask_img.save(mask_path)

        regions_in_mask = sum(
            1 for r in regions if args.include_sfx or r.get("keep", True)
        )
        total_regions_masked += regions_in_mask

        print(f"PROGRESS {idx}/{len(image_files)} {filename} regions_masked={regions_in_mask}", flush=True)

    print(f"DONE total_images={len(image_files)} total_regions_masked={total_regions_masked} skipped_no_json={total_skipped_no_json}", flush=True)


if __name__ == "__main__":
    main()
'''

SCRIPT_PATH = '/content/run_inpaint.py'
with open(SCRIPT_PATH, 'w', encoding='utf-8') as f:
    f.write(inpaint_script)

print(f'Đã ghi script inpaint tại: {SCRIPT_PATH}')

## 5.3 Cấu hình inpaint

In [ ]:
# False (mặc định): chỉ xóa vùng thoại (keep=True), GIỮ NGUYÊN SFX gốc.
INCLUDE_SFX = False

# Số pixel giãn mask thêm quanh box OCR, để xóa sạch viền chữ mờ còn sót.
DILATE_PX = 3

TEST_LIMIT = 0  # 0 = chạy toàn bộ chapter (mặc định khi chạy full pipeline)

print(f'Đọc ảnh gốc từ: {RAW_DIR}')
print(f'Đọc JSON OCR từ: {OCR_OUTPUT_DIR}')
print(f'Lưu ảnh đã xóa chữ tại: {INPAINTED_DIR}')
print(f'Lưu mask (debug) tại: {MASK_DIR}')
print(f'Xóa cả SFX: {INCLUDE_SFX}')

## 5.4 Chạy inpaint (thực thi trong venv qua subprocess)

Lần chạy đầu tiên sẽ mất thêm chút thời gian để script tự tải checkpoint LaMa (~200MB) — checkpoint được lưu trong venv (hoặc cache mặc định của thư viện trên ổ tạm), không phải trên Drive, nên vẫn cần tải lại nếu bạn tạo venv mới ở phiên Colab sau. Xử lý mỗi trang cũng chậm hơn OCR khá nhiều (LaMa là model nặng hơn), đặc biệt nếu Colab không cấp GPU.

In [ ]:
import subprocess
import time

cmd = [
    VENV_PY, SCRIPT_PATH,
    '--raw_dir', RAW_DIR,
    '--ocr_dir', OCR_OUTPUT_DIR,
    '--output_dir', INPAINTED_DIR,
    '--mask_dir', MASK_DIR,
    '--dilate_px', str(DILATE_PX),
    '--limit', str(TEST_LIMIT),
]
if INCLUDE_SFX:
    cmd.append('--include_sfx')

start_time = time.time()

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in process.stdout:
    print(line, end='')  # in trực tiếp log từ venv ra notebook, bao gồm dòng PROGRESS/DONE

process.wait()
elapsed = time.time() - start_time

if process.returncode != 0:
    print(f"\\nScript inpaint thoát với lỗi (mã {process.returncode}). Xem log phía trên để biết chi tiết.")
else:
    print(f"\\nHoàn tất inpaint trong {elapsed:.1f} giây.")

## 5.5 Xem trước kết quả (gốc / mask / đã xóa chữ)

So sánh 3 cột cho mỗi trang: ảnh gốc, mask (vùng trắng = đã xóa), và ảnh sau khi inpaint — giúp đánh giá nhanh `DILATE_PX` đã hợp lý chưa (viền chữ còn sót hay mask lấn quá nhiều vào tranh nền).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])

PREVIEW_COUNT = 3
preview_files = image_files[:PREVIEW_COUNT]

fig, axes = plt.subplots(len(preview_files), 3, figsize=(15, 6 * len(preview_files)))
if len(preview_files) == 1:
    axes = [axes]

for row, filename in zip(axes, preview_files):
    raw_path = os.path.join(RAW_DIR, filename)
    mask_path = os.path.join(MASK_DIR, os.path.splitext(filename)[0] + '_mask.png')
    inpainted_path = os.path.join(INPAINTED_DIR, filename)

    row[0].imshow(Image.open(raw_path))
    row[0].set_title(f'{filename}\n(gốc)', fontsize=9)
    row[0].axis('off')

    if os.path.exists(mask_path):
        row[1].imshow(Image.open(mask_path), cmap='gray')
        row[1].set_title('mask (trắng = đã xóa)', fontsize=9)
    else:
        row[1].set_title('(không có mask)', fontsize=9)
    row[1].axis('off')

    if os.path.exists(inpainted_path):
        row[2].imshow(Image.open(inpainted_path))
        row[2].set_title('sau khi inpaint', fontsize=9)
    else:
        row[2].set_title('(chưa xử lý)', fontsize=9)
    row[2].axis('off')

plt.tight_layout()
plt.show()

---
# BƯỚC 6: Giao diện Gradio — chỉnh sửa bản dịch & chèn chữ vào ảnh

Notebook này mở một giao diện web (Gradio) để bạn:
1. Xem từng trang đã xóa chữ (từ Bước 4) song song với danh sách bản dịch (từ Bước 3)
2. Sửa lại câu chữ tiếng Việt, tick bỏ vùng không cần (SFX, v.v.)
3. Bật/tắt in đậm, chỉnh cỡ chữ nếu cần
4. Xem preview ảnh đã chèn chữ ngay trong giao diện
5. Lưu ảnh hoàn chỉnh ra thư mục `final/`

**Font chữ:** dùng DejaVu Sans có sẵn trên Colab (hỗ trợ đầy đủ dấu tiếng Việt), không cần tải thêm font ngoài. Có nút bật in đậm và slider chỉnh cỡ chữ thủ công cho từng vùng.

**Lưu ý quan trọng:** Gradio chạy như một server sống trong cell — cell chạy server sẽ **không tự kết thúc**, notebook cần được giữ mở trong lúc bạn thao tác. Dùng `share=True` để có link truy cập từ trình duyệt bất kỳ (không chỉ trong Colab), link này có hiệu lực khoảng 72 giờ.

**Quy trình:**
1. Kết nối Google Drive
2. Cài Gradio
3. Cấu hình
4. Các hàm xử lý: render chữ lên ảnh, load/save trang
5. Dựng giao diện Gradio và khởi chạy

## 6.1 Cài Gradio + font

Gradio chạy trực tiếp trong kernel Colab (không cần venv riêng như các bước trước) — nó chỉ là thư viện Python thuần, không có xung đột dependency phức tạp như PaddleOCR/LaMa.

Font `fonts-dejavu-core` (hỗ trợ đầy đủ dấu tiếng Việt) không phải lúc nào cũng có sẵn trên mọi máy ảo Colab, nên cài tường minh ở đây để chắc chắn — tránh phải chạy lệnh tay khi gặp lỗi thiếu font.

In [ ]:
!pip install -q gradio
!apt-get install -y -qq fonts-dejavu-core > /dev/null 2>&1

## 6.2 Cấu hình font

In [ ]:
# Font hệ thống có sẵn trên Colab, hỗ trợ đầy đủ dấu tiếng Việt.
FONT_REGULAR_PATH = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
FONT_BOLD_PATH = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'

# Một số máy ảo Colab không có sẵn font ở đường dẫn mặc định — tự dò
# thêm nếu không tìm thấy, tránh phải sửa tay đường dẫn.
if not os.path.exists(FONT_REGULAR_PATH):
    found = !find / -iname 'DejaVuSans.ttf' 2>/dev/null
    if found:
        FONT_REGULAR_PATH = found[0]
        FONT_BOLD_PATH = found[0].replace('DejaVuSans.ttf', 'DejaVuSans-Bold.ttf')

for p in (FONT_REGULAR_PATH, FONT_BOLD_PATH):
    if not os.path.exists(p):
        raise RuntimeError(
            f'Không tìm thấy font tại {p}. '
            'Chạy lại cell mục 5.1 (cài fonts-dejavu-core), hoặc chạy tay: '
            '!apt-get install -y fonts-dejavu-core, rồi chạy lại cell này.'
        )

print(f'Đọc ảnh gốc từ: {RAW_DIR}')
print(f'Đọc/ghi JSON dịch tại: {OCR_OUTPUT_DIR}')
print(f'Đọc ảnh đã xóa chữ từ: {INPAINTED_DIR}')
print(f'Lưu ảnh hoàn chỉnh tại: {FINAL_DIR}')

## 6.3 Các hàm xử lý chính

- `list_pages()`: liệt kê danh sách trang theo đúng thứ tự (dựa vào `INPAINTED_DIR`, khớp thứ tự với `RAW_DIR`/Bước 1)
- `load_page_data()`: đọc JSON của 1 trang
- `render_page()`: vẽ chữ Việt lên ảnh đã inpaint theo từng box, tự wrap dòng và tự co cỡ chữ vừa khung (trừ khi người dùng override cỡ chữ thủ công)
- `save_page()`: lưu ảnh render ra `FINAL_DIR` + ghi lại JSON (để giữ lại chỉnh sửa)

In [ ]:
import json
import textwrap
from PIL import Image, ImageDraw, ImageFont


def list_pages():
    """Danh sach ten file trang, lay tu INPAINTED_DIR de dam bao chi lay
    nhung trang da qua Buoc 4. Sap xep theo ten (da co prefix so thu tu
    tu Buoc 1 nen thu tu luon dung)."""
    if not os.path.isdir(INPAINTED_DIR):
        return []
    return sorted([
        f for f in os.listdir(INPAINTED_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
    ])


def json_path_for(filename):
    return os.path.join(OCR_OUTPUT_DIR, os.path.splitext(filename)[0] + '.json')


def load_page_data(filename):
    """Doc JSON cua 1 trang. Neu chua co file (truong hop la) thi tra ve
    cau truc rong de UI khong bi crash."""
    path = json_path_for(filename)
    if not os.path.exists(path):
        return {'source_image': filename, 'regions': []}
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_page_data(filename, data):
    path = json_path_for(filename)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def _box_bounds(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    return min(xs), min(ys), max(xs), max(ys)


def _wrap_by_pixel_width(font, text, max_width):
    """Wrap text theo tung TU, do bang chieu rong pixel thuc te cua font
    (khong uoc luong theo so ky tu). Chinh xac hon textwrap.fill vi chu
    tieng Viet co dau rong/hep khong deu nhau giua cac ky tu, uoc luong
    theo do dai ky tu 'A' truoc day gay wrap sai -> chu de chong len nhau.
    """
    words = text.split()
    if not words:
        return ['']

    lines = []
    current = words[0]
    for word in words[1:]:
        candidate = current + ' ' + word
        if font.getlength(candidate) <= max_width:
            current = candidate
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines


def _fit_text_in_box(draw, text, box_w, box_h, bold, manual_size):
    """Tim cach wrap + cỡ chu de text vua khop trong khung box.

    Neu manual_size > 0: day la CO CHU TOI DA mong muon (de dong bo giua
    cac vung). Neu vua khop box o dung cỡ do thi dung luon -- giup da so
    bong bong deu dong bo cung 1 cỡ chu. Nhung neu bong bong qua nho so
    voi cỡ do (VD bong bong nho, cau dai), tu dong giam dan cỡ CHI RIENG
    vung nay xuong toi khi vua khop, thay vi giu nguyen va de chu tran/
    de len bong bong ben canh. Day la ly do gay loi chong chu khi ban
    chinh cỡ 35 tro len o mot so trang.

    Neu manual_size == 0 (tu dong hoan toan): thu giam dan cỡ chu tu lon
    xuong nho cho tung vung, khong co muc uu tien nao.
    """
    font_path = FONT_BOLD_PATH if bold else FONT_REGULAR_PATH

    def try_size(size):
        font = ImageFont.truetype(font_path, size)
        # Gioi han chieu rong wrap con 92% box de chua vien trang quanh
        # chu (ve o buoc sau) khong bi tran ra ngoai canh box.
        lines = _wrap_by_pixel_width(font, text, box_w * 0.92)
        wrapped = '\n'.join(lines)
        line_height = size * 1.3
        total_h = line_height * len(lines)
        max_line_w = max((font.getlength(line) for line in lines), default=0)
        return font, wrapped, lines, line_height, total_h, max_line_w

    start_size = int(manual_size) if manual_size and manual_size > 0 else 48
    min_size = 7

    for size in range(start_size, min_size - 1, -1):
        font, wrapped, lines, line_height, total_h, max_line_w = try_size(size)
        if total_h <= box_h and max_line_w <= box_w:
            return font, wrapped, lines, line_height, total_h, max_line_w

    return try_size(min_size)


def render_page(filename, edited_regions, bold, manual_size):
    """Ve chu Viet len anh da inpaint, theo box cua tung region con keep=True.

    edited_regions: danh sach dict co it nhat 'box', 'text_translated', 'keep'
    (lay tu cac o input tren giao dien, khong doc lai tu JSON de phan anh
    dung thay doi nguoi dung vua go, chua luu).
    """
    inpainted_path = os.path.join(INPAINTED_DIR, filename)
    img = Image.open(inpainted_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    for region in edited_regions:
        if not region.get('keep', True):
            continue
        text = (region.get('text_translated') or '').strip()
        if not text:
            continue

        box = region['box']
        x0, y0, x1, y1 = _box_bounds(box)
        box_w, box_h = max(1, x1 - x0), max(1, y1 - y0)

        font, wrapped, lines, line_height, total_h, max_line_w = _fit_text_in_box(
            draw, text, box_w, box_h, bold, manual_size
        )

        start_y = y0 + max(0, (box_h - total_h) / 2)
        for i, line in enumerate(lines):
            line_w = font.getlength(line)
            line_x = x0 + max(0, (box_w - line_w) / 2)
            line_y = start_y + i * line_height
            # Vien trang mong quanh chu den giup chu de doc hon tren nen
            # anh vua duoc inpaint (co the co texture/mau khong deu).
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx or dy:
                        draw.text((line_x + dx, line_y + dy), line, font=font, fill='white')
            draw.text((line_x, line_y), line, font=font, fill='black')

    return img


print('Đã định nghĩa xong các hàm xử lý.')

## 6.4 Dựng giao diện Gradio

Bố cục: ảnh preview bên trái, danh sách vùng chữ (tối đa 20 vùng/trang — đủ dùng cho hầu hết trang manga, có thể tăng `MAX_REGIONS` nếu trang nào có nhiều thoại hơn) bên phải, mỗi vùng có ô sửa text + checkbox `keep`. Nút điều hướng trang ở trên cùng.

In [ ]:
import gradio as gr

MAX_REGIONS = 20  # so o text toi da hien thi tren 1 trang; tang neu trang co nhieu thoai hon

pages = list_pages()
if not pages:
    raise RuntimeError(
        f'Không tìm thấy trang nào trong {INPAINTED_DIR}. '
        'Kiểm tra lại CHAPTER_ID hoặc chạy Bước 4 (inpaint) trước.'
    )
print(f'Tìm thấy {len(pages)} trang.')


def _page_state_to_ui(page_idx, bold=False, manual_size=20):
    """Doc du lieu 1 trang tu JSON, tra ve cac gia tri de dien vao UI.

    Nhan them bold/manual_size de preview luc chuyen trang dung DUNG cung
    cai dat font nguoi dung dang chinh, thay vi luon auto-fit rieng tung
    vung (nguyen nhan gay chu to nho khong deu giua cac trang).
    """
    filename = pages[page_idx]
    data = load_page_data(filename)
    regions = data.get('regions', [])

    text_updates = []
    keep_updates = []
    original_texts = []
    row_visibility = []

    for i in range(MAX_REGIONS):
        if i < len(regions):
            r = regions[i]
            text_updates.append(gr.update(value=r.get('text_translated', '')))
            keep_updates.append(gr.update(value=r.get('keep', True)))
            original_texts.append(f"**Gốc:** {r.get('text_original', '')}")
            row_visibility.append(gr.update(visible=True))
        else:
            text_updates.append(gr.update(value=''))
            keep_updates.append(gr.update(value=True))
            original_texts.append('')
            row_visibility.append(gr.update(visible=False))

    preview_img = render_page(filename, regions, bold=bold, manual_size=int(manual_size))
    status = f'Trang {page_idx + 1}/{len(pages)} — {filename} ({len(regions)} vùng chữ)'

    return [preview_img, status] + text_updates + keep_updates + original_texts + row_visibility


def _collect_regions_from_ui(page_idx, *args):
    """Ghep lai danh sach region tu cac o input hien tai tren UI + box goc
    da luu trong JSON (box khong doi trong Buoc 5, chi text/keep doi)."""
    filename = pages[page_idx]
    data = load_page_data(filename)
    regions = data.get('regions', [])

    texts = args[:MAX_REGIONS]
    keeps = args[MAX_REGIONS:2 * MAX_REGIONS]

    updated_regions = []
    for i, r in enumerate(regions):
        new_r = dict(r)
        if i < MAX_REGIONS:
            new_r['text_translated'] = texts[i]
            new_r['keep'] = bool(keeps[i])
        updated_regions.append(new_r)

    return filename, data, updated_regions


def on_preview(page_idx, bold, manual_size, *args):
    filename, _, updated_regions = _collect_regions_from_ui(page_idx, *args)
    img = render_page(filename, updated_regions, bold=bold, manual_size=int(manual_size))
    return img


def on_save(page_idx, bold, manual_size, *args):
    filename, data, updated_regions = _collect_regions_from_ui(page_idx, *args)
    data['regions'] = updated_regions
    save_page_data(filename, data)

    final_img = render_page(filename, updated_regions, bold=bold, manual_size=int(manual_size))
    final_path = os.path.join(FINAL_DIR, filename)
    final_img.save(final_path)
    return final_img, f'Đã lưu: {final_path}'


def on_nav(page_idx, direction, bold, manual_size):
    new_idx = max(0, min(len(pages) - 1, page_idx + direction))
    return [new_idx] + _page_state_to_ui(new_idx, bold, manual_size)


def on_jump(page_number, bold, manual_size):
    new_idx = max(0, min(len(pages) - 1, int(page_number) - 1))
    return [new_idx] + _page_state_to_ui(new_idx, bold, manual_size)


with gr.Blocks(title='Chỉnh sửa bản dịch manga') as demo:
    gr.Markdown('# Chỉnh sửa bản dịch & chèn chữ vào ảnh')

    page_idx_state = gr.State(0)

    with gr.Row():
        btn_prev = gr.Button('◀ Trang trước')
        page_jump = gr.Number(value=1, label='Đi tới trang', precision=0)
        btn_jump = gr.Button('Đi tới')
        btn_next = gr.Button('Trang sau ▶')

    status_text = gr.Markdown()

    with gr.Row():
        with gr.Column(scale=1):
            image_preview = gr.Image(label='Preview ảnh đã chèn chữ', type='pil')
            with gr.Row():
                bold_toggle = gr.Checkbox(label='In đậm', value=False)
                font_size_slider = gr.Slider(
                    label='Cỡ chữ (0 = tự động co vừa khung, nên để cố định để đồng bộ cả trang)',
                    minimum=0, maximum=60, step=1, value=20
                )
            with gr.Row():
                btn_preview = gr.Button('🔄 Render preview')
                btn_save = gr.Button('💾 Lưu trang này', variant='primary')

        with gr.Column(scale=1):
            gr.Markdown('### Các vùng chữ trên trang')
            text_boxes = []
            keep_boxes = []
            original_labels = []
            region_rows = []
            for i in range(MAX_REGIONS):
                with gr.Group(visible=False) as row:
                    orig_label = gr.Markdown()
                    with gr.Row():
                        txt = gr.Textbox(label=f'Vùng {i + 1} — bản dịch', lines=2)
                        keep_chk = gr.Checkbox(label='Giữ', value=True)
                region_rows.append(row)
                original_labels.append(orig_label)
                text_boxes.append(txt)
                keep_boxes.append(keep_chk)

    all_ui_outputs = [image_preview, status_text] + text_boxes + keep_boxes + original_labels + region_rows
    all_region_inputs = text_boxes + keep_boxes

    demo.load(
        fn=lambda bold, size: _page_state_to_ui(0, bold, size),
        inputs=[bold_toggle, font_size_slider],
        outputs=all_ui_outputs,
    )

    btn_prev.click(
        fn=lambda idx, bold, size: on_nav(idx, -1, bold, size),
        inputs=[page_idx_state, bold_toggle, font_size_slider],
        outputs=[page_idx_state] + all_ui_outputs,
    )
    btn_next.click(
        fn=lambda idx, bold, size: on_nav(idx, 1, bold, size),
        inputs=[page_idx_state, bold_toggle, font_size_slider],
        outputs=[page_idx_state] + all_ui_outputs,
    )
    btn_jump.click(
        fn=on_jump,
        inputs=[page_jump, bold_toggle, font_size_slider],
        outputs=[page_idx_state] + all_ui_outputs,
    )

    btn_preview.click(
        fn=on_preview,
        inputs=[page_idx_state, bold_toggle, font_size_slider] + all_region_inputs,
        outputs=[image_preview],
    )
    btn_save.click(
        fn=on_save,
        inputs=[page_idx_state, bold_toggle, font_size_slider] + all_region_inputs,
        outputs=[image_preview, status_text],
    )

demo.launch(share=True, debug=True)  # debug=True QUAN TRONG: giu cell nay "block" khi bam Run All,
# tranh truong hop pipeline tu chay tiep xuong Buoc 7 (dong goi .cbz)
# va Buoc 7.5 (don dep, CO THE XOA ANH GOC) truoc khi ban kip sua o Gradio.

---
## Xong bước 5

**Cách dùng:**
1. Chạy hết các cell từ 1 đến 5
2. Cell cuối sẽ in ra 2 link: một link local (chỉ dùng được trong Colab) và một link `https://xxxxx.gradio.live` — mở link `.gradio.live` bằng trình duyệt bất kỳ
3. Với mỗi trang: sửa text trong các ô "bản dịch", bỏ tick "Giữ" cho vùng không cần (VD SFX), bấm **Render preview** để xem thử, chỉnh **In đậm** / **Cỡ chữ** nếu cần, rồi bấm **Lưu trang này**
4. Dùng nút **Trang trước / Trang sau** hoặc ô **Đi tới trang** để duyệt qua cả chapter

**Ảnh hoàn chỉnh** được lưu tại `FINAL_DIR` (`chapters/{CHAPTER_ID}/final/`), sẵn sàng để ghép lại thành file truyện hoàn chỉnh.

**Lưu ý:**
- Bấm **Lưu trang này** cũng đồng thời ghi lại `text_translated`/`keep` vào JSON gốc trong `ocr_results/` — nên nếu bạn thoát giữa chừng rồi quay lại, các trang đã sửa vẫn giữ nguyên nội dung đã sửa (không bị mất)
- Nếu trang nào có nhiều hơn 20 vùng chữ, tăng biến `MAX_REGIONS` ở đầu mục 5 rồi chạy lại
- Cỡ chữ tự động (giá trị 0) sẽ cố co chữ vừa khung box gốc; nếu câu tiếng Việt quá dài so với khung (tiếng Việt thường dài hơn tiếng Anh/Hàn), chữ sẽ tự nhỏ dần tới khi vừa — nếu vẫn thấy quá nhỏ, cân nhắc rút gọn câu dịch
- Đóng tab trình duyệt không tắt server — muốn dừng hẳn, ngắt (interrupt) cell đang chạy `demo.launch()` trong Colab

---
# BƯỚC 7: Xuất chapter hoàn chỉnh ra file `.cbz`

Notebook đóng gói toàn bộ ảnh đã typeset xong (trong `FINAL_DIR`, kết quả Bước 6) thành 1 file `.cbz` duy nhất — định dạng chuẩn để đọc bằng các app đọc truyện (Komga, Tachiyomi, CDisplayEx, YACReader...). `.cbz` thực chất chỉ là file `.zip` chứa ảnh theo thứ tự, đổi đuôi thành `.cbz`.

**Quy trình:**
1. Lấy tên/số chapter thật từ MangaDex (qua API `/chapter/{id}`), cho phép sửa tay nếu muốn
2. Đóng gói ảnh trong `FINAL_DIR` thành `.cbz`
3. Kiểm tra lại file `.cbz` vừa tạo (mở thử, đếm số ảnh) để chắc chắn không hỏng


## 7.0 Kiểm tra an toàn — đảm bảo đã hoàn thành Bước 6 (Gradio) cho TOÀN BỘ trang

**Vì sao cần bước này:** nếu bạn bấm "Run all"/"Run below" hoặc lỡ ngắt cell Gradio (Bước 6) trước khi sửa xong hết trang, các bước sau (đóng gói `.cbz`, và đặc biệt là mục 7.5 dọn dẹp — CÓ THỂ XOÁ ẢNH GỐC) sẽ chạy tiếp với dữ liệu thiếu/sai mà không cảnh báo gì. Cell dưới đây chặn lại ngay nếu phát hiện thiếu trang.

In [ ]:
raw_page_names = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])
final_page_names = set(os.listdir(FINAL_DIR)) if os.path.isdir(FINAL_DIR) else set()

missing_pages = [f for f in raw_page_names if f not in final_page_names]

print(f"Tổng số trang gốc:        {len(raw_page_names)}")
print(f"Số trang đã lưu ở Bước 6: {len(final_page_names)}")

if missing_pages:
    raise RuntimeError(
        f"CÒN THIẾU {len(missing_pages)}/{len(raw_page_names)} TRANG chưa được xử lý ở Bước 6 (Gradio) — "
        f"chưa thể đóng gói .cbz.\n"
        f"Các trang còn thiếu: {missing_pages}\n\n"
        "Quay lại chạy cell Gradio ở Bước 6, mở link .gradio.live, duyệt tới đúng các trang "
        "còn thiếu ở trên, bấm 'Lưu trang này' cho từng trang, rồi quay lại chạy từ đầu Bước 7."
    )

print(f"\nOK — đủ {len(raw_page_names)}/{len(raw_page_names)} trang đã qua Bước 6. Có thể tiếp tục đóng gói .cbz.")


## 7.1 Lấy tên/số chapter từ MangaDex

Gọi API `GET /chapter/{id}` (khác với API `/at-home/server/{id}` đã dùng ở Bước 1 để lấy ảnh) để lấy số chapter (`chapter`) và tiêu đề (`title`) do người upload đặt trên MangaDex. Hai trường này không phải lúc nào cũng có (một số chapter không đặt title, hoặc số chapter để trống với oneshot), nên có fallback hợp lý cho từng trường hợp.

In [ ]:
import requests


def get_chapter_metadata(chapter_id: str) -> dict:
    """Goi API MangaDex de lay so chapter + title. Khac voi ham
    get_chapter_server_info() o Buoc 1 (dung de lay anh) — ham nay chi
    lay metadata, khong lien quan toi viec tai anh.
    """
    url = f"https://api.mangadex.org/chapter/{chapter_id}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return resp.json()["data"]["attributes"]


try:
    chapter_meta = get_chapter_metadata(CHAPTER_ID)
    chapter_number = chapter_meta.get("chapter")  # vd: "12", co the None neu oneshot
    chapter_title = chapter_meta.get("title")      # vd: "Inside A Taxi", co the None
except requests.RequestException as e:
    print(f"Khong lay duoc metadata tu MangaDex ({e}). Se dung ten mac dinh theo CHAPTER_ID.")
    chapter_number = None
    chapter_title = None

print(f"So chapter: {chapter_number!r}")
print(f"Tieu de: {chapter_title!r}")


## 7.2 Cấu hình tên file `.cbz`

Tên file được tự sinh theo mẫu `Chapter_{số}` (kèm `_{tiêu đề}` nếu MangaDex có title), nhưng bạn có thể ghi đè hoàn toàn bằng `CBZ_FILENAME_OVERRIDE` — hữu ích khi số chapter bị thiếu, hoặc bạn muốn đặt tên khác cho khớp bộ sưu tập của mình.

In [ ]:
import re


def _slugify_for_filename(text: str) -> str:
    """Loai bo ky tu khong hop le trong ten file (giu chu/so/khoang trang/
    gach ngang/gach duoi), rut gon khoang trang thua."""
    text = re.sub(r'[\\/:*?"<>|]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Đặt tay tên file ở đây nếu muốn override hoàn toàn (không kèm đuôi .cbz,
# notebook tự thêm vào). Để None để tự sinh tên theo chapter_number/chapter_title
# lấy được ở mục 7.1.
CBZ_FILENAME_OVERRIDE = None

if CBZ_FILENAME_OVERRIDE:
    cbz_filename = _slugify_for_filename(CBZ_FILENAME_OVERRIDE) + '.cbz'
else:
    if chapter_number:
        name_parts = [f"Chapter_{chapter_number}"]
    else:
        # Khong co so chapter (vd oneshot) -> dung 8 ky tu dau CHAPTER_ID cho de nhan biet
        name_parts = [f"Chapter_{CHAPTER_ID[:8]}"]

    if chapter_title:
        name_parts.append(_slugify_for_filename(chapter_title))

    cbz_filename = _slugify_for_filename('_'.join(name_parts)) + '.cbz'

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'chapters', CHAPTER_ID, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
CBZ_PATH = os.path.join(OUTPUT_DIR, cbz_filename)

print(f"Ten file .cbz se tao: {cbz_filename}")
print(f"Duong dan day du: {CBZ_PATH}")


## 7.3 Đóng gói ảnh thành `.cbz`

Đọc toàn bộ ảnh trong `FINAL_DIR` (kết quả Bước 6, đã chỉnh sửa và chèn chữ xong), sắp theo đúng thứ tự tên file (đã có prefix số thứ tự từ Bước 1 nên thứ tự luôn đúng), rồi nén thành `.cbz`.

In [ ]:
import zipfile

final_images = sorted([
    f for f in os.listdir(FINAL_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])

if not final_images:
    raise RuntimeError(
        f"Khong tim thay anh nao trong {FINAL_DIR}. "
        "Kiem tra lai da chay xong Buoc 6 (Gradio) va da bam 'Luu trang nay' cho tung trang chua."
    )

print(f"Tim thay {len(final_images)} trang trong FINAL_DIR, chuan bi dong goi...")

if os.path.exists(CBZ_PATH):
    print(f"Da co file .cbz cu cung ten, se ghi de: {CBZ_PATH}")

with zipfile.ZipFile(CBZ_PATH, 'w', zipfile.ZIP_STORED) as cbz:
    for filename in final_images:
        file_path = os.path.join(FINAL_DIR, filename)
        # arcname = ten file trong zip, dung dung ten goc de giu thu tu trang
        cbz.write(file_path, arcname=filename)

cbz_size_mb = os.path.getsize(CBZ_PATH) / (1024 * 1024)
print(f"Da tao xong: {CBZ_PATH} ({cbz_size_mb:.1f} MB, {len(final_images)} trang)")


## 7.4 Kiểm tra lại file vừa tạo

Mở lại file `.cbz` bằng `zipfile` (test tính toàn vẹn của zip) và đối chiếu số ảnh bên trong với số ảnh ở `FINAL_DIR`, để chắc chắn file không bị hỏng hoặc thiếu trang trước khi bạn đem đi dùng.

In [ ]:
with zipfile.ZipFile(CBZ_PATH, 'r') as cbz:
    bad_file = cbz.testzip()
    names_in_cbz = cbz.namelist()

if bad_file is not None:
    raise RuntimeError(f"File .cbz bi hong o entry: {bad_file}. Chay lai muc 7.3.")

if len(names_in_cbz) != len(final_images):
    print(
        f"CANH BAO: so anh trong .cbz ({len(names_in_cbz)}) khac so anh trong "
        f"FINAL_DIR ({len(final_images)}). Kiem tra lai."
    )
else:
    print(f"OK — file .cbz hop le, du {len(names_in_cbz)} trang, khop voi FINAL_DIR.")

print(f"\nFile san sang tai: {CBZ_PATH}")


---
## 7.5 (Tuỳ chọn) Dọn dẹp thư mục trung gian

**Không chạy tự động** — mục này để riêng, bạn tự bấm chạy khi thấy Drive đầy hoặc muốn giải phóng dung lượng sau khi đã có `.cbz` hoàn chỉnh.

Mỗi chapter hiện giữ nhiều bản sao ảnh trong các thư mục: `raw/` (ảnh gốc), `inpainted/` (đã xoá chữ), `masks/`, `final/` (đã chèn chữ Việt), `ocr_results_backup_before_merge/` — nếu dịch nhiều chapter, tổng dung lượng này tăng rất nhanh trong khi file cần giữ lại chỉ là `.cbz` ở `output/`.

**An toàn:** cell dưới đây **luôn kiểm tra `.cbz` đã tồn tại và hợp lệ trước khi xoá bất cứ gì** — nếu chưa có `.cbz` hợp lệ, nó sẽ dừng lại và báo lỗi thay vì xoá nhầm.

In [ ]:
import shutil

# CÔNG TẮC AN TOÀN: mặc định là None (KHÔNG HỢP LỆ) — cell này CHỦ Ý không
# chạy được nếu bạn không tự tay sửa dòng dưới. Đây là để chống trường hợp
# bấm "Run all"/"Run below" lỡ lướt qua cell này và xoá nhầm ảnh gốc, đúng
# như tình huống đã từng xảy ra (Gradio debug=False khiến Run All tự trôi
# xuống tới tận đây). CHỈ đổi CLEANUP_MODE khi bạn ĐÃ CHỦ ĐỘNG quyết định
# dọn dẹp, và đã đọc kỹ 2 lựa chọn bên dưới.
#
# - "full"       : xoá HẾT thư mục trung gian (raw, ocr_results, inpainted,
#                   masks, final...), chỉ giữ lại output/*.cbz. Dùng khi bạn
#                   chắc chắn không cần dịch/sửa lại chapter này nữa.
# - "keep_ocr"   : chỉ xoá phần ẢNH (raw, inpainted, masks, final — nặng),
#                   giữ lại ocr_results/ (JSON nhẹ, chứa text gốc + bản dịch).
#                   Sau này muốn sửa lại bản dịch hoặc render lại ảnh vẫn
#                   được, nhưng phải OCR lại từ raw nếu cần đổi ảnh gốc.
CLEANUP_MODE = None # PHẢI tự tay đổi thành "full" hoặc "keep_ocr" mới chạy được

if CLEANUP_MODE not in ("full", "keep_ocr"):
    raise RuntimeError(
        "CLEANUP_MODE đang là None (giá trị mặc định, cố ý không hợp lệ). "
        "Đây là công tắc an toàn để mục dọn dẹp không tự chạy khi bấm 'Run all'. "
        "Nếu bạn THỰC SỰ muốn dọn dẹp thư mục trung gian của chapter này, "
        "sửa dòng CLEANUP_MODE ở trên thành \"full\" hoặc \"keep_ocr\", "
        "rồi chạy lại RIÊNG cell này."
    )

# ----- Kiểm tra an toàn: bắt buộc phải có .cbz hợp lệ trước khi xoá -----
if not os.path.exists(CBZ_PATH):
    raise RuntimeError(
        f"Chưa tìm thấy file .cbz tại {CBZ_PATH}. "
        "Chạy xong mục 7.3 và 7.4 trước, đảm bảo .cbz đã tạo và kiểm tra OK, "
        "rồi mới chạy dọn dẹp."
    )

with zipfile.ZipFile(CBZ_PATH, 'r') as cbz:
    if cbz.testzip() is not None:
        raise RuntimeError(
            f"File .cbz tại {CBZ_PATH} bị hỏng (testzip thất bại). "
            "Chạy lại mục 7.3 để tạo lại .cbz trước khi dọn dẹp."
        )

print(f"Đã xác nhận .cbz hợp lệ tại: {CBZ_PATH}")

# ----- Xác định danh sách thư mục cần xoá theo chế độ đã chọn -----
if CLEANUP_MODE == "full":
    dirs_to_delete = [RAW_DIR, OCR_OUTPUT_DIR, BACKUP_DIR, INPAINTED_DIR, MASK_DIR, FINAL_DIR]
elif CLEANUP_MODE == "keep_ocr":
    dirs_to_delete = [RAW_DIR, BACKUP_DIR, INPAINTED_DIR, MASK_DIR, FINAL_DIR]
else:
    raise ValueError(f"CLEANUP_MODE không hợp lệ: {CLEANUP_MODE!r} (chỉ nhận 'full' hoặc 'keep_ocr')")


def _dir_size_mb(path):
    if not os.path.isdir(path):
        return 0.0
    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path)
        for f in files
    )
    return total / (1024 * 1024)


total_freed_mb = 0.0
for d in dirs_to_delete:
    if os.path.isdir(d):
        size_mb = _dir_size_mb(d)
        shutil.rmtree(d)
        total_freed_mb += size_mb
        print(f"Đã xoá: {d} ({size_mb:.1f} MB)")
    else:
        print(f"Bỏ qua (không tồn tại): {d}")

print(f"\nTổng dung lượng đã giải phóng: {total_freed_mb:.1f} MB")
print(f"File .cbz vẫn còn nguyên tại: {CBZ_PATH}")
if CLEANUP_MODE == "keep_ocr":
    print(f"Đã giữ lại: {OCR_OUTPUT_DIR} (JSON bản dịch, có thể sửa/render lại sau)")


---
## Xong bước 7 — Xong toàn bộ pipeline

File `.cbz` hoàn chỉnh của chapter nằm tại `chapters/{CHAPTER_ID}/output/{tên file}.cbz` trên Drive, sẵn sàng mở bằng bất kỳ app đọc truyện nào hỗ trợ `.cbz` (Komga, Tachiyomi, CDisplayEx, YACReader, v.v.), hoặc up lên nơi bạn muốn lưu trữ.

**Chạy cho chapter tiếp theo:** quay lại cell đầu tiên (mục 1, khai báo `CHAPTER_ID`), đổi sang ID chapter mới, rồi chạy lại toàn bộ pipeline từ Bước 1 đến Bước 7.

**Gợi ý dọn dẹp (tùy chọn):** các thư mục trung gian (`ocr_results_backup_before_merge`, `masks`) không cần thiết cho việc đọc truyện, có thể xoá để tiết kiệm dung lượng Drive nếu muốn, chỉ cần giữ lại `output/` (file `.cbz` cuối) — hoặc giữ nguyên nếu muốn có thể quay lại chỉnh sửa/dịch lại sau này.